### Consumer Pricing

Transactional records w/ premise and items

- https://data.gov.my/data-catalogue/pricecatcher
- https://data.gov.my/data-catalogue/lookup_item
- https://data.gov.my/data-catalogue/lookup_premise


In [8]:
%pip install numpy
%pip install pandas
%pip install matplotlib
%pip install seaborn
%pip install sklearn
%pip install scipy
%pip install pyarrow
%pip install fastparquet

Defaulting to user installation because normal site-packages is not writeable
You should consider upgrading via the '/Library/Developer/CommandLineTools/usr/bin/python3 -m pip install --upgrade pip' command.
Note: you may need to restart the kernel to use updated packages.
Defaulting to user installation because normal site-packages is not writeable
You should consider upgrading via the '/Library/Developer/CommandLineTools/usr/bin/python3 -m pip install --upgrade pip' command.
Note: you may need to restart the kernel to use updated packages.
Defaulting to user installation because normal site-packages is not writeable
You should consider upgrading via the '/Library/Developer/CommandLineTools/usr/bin/python3 -m pip install --upgrade pip' command.
Note: you may need to restart the kernel to use updated packages.
Defaulting to user installation because normal site-packages is not writeable
You should consider upgrading via the '/Library/Developer/CommandLineTools/usr/bin/python3 -m pip in

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os

ModuleNotFoundError: No module named 'pyarrow'

In [6]:
# Source URLs from the catalog metadata pages
TRANSACTION_URL = "https://storage.data.gov.my/pricecatcher/pricecatcher_2026-02.parquet"
ITEM_URL = "https://storage.data.gov.my/pricecatcher/lookup_item.parquet"
PREMISE_URL = "https://storage.data.gov.my/pricecatcher/lookup_premise.parquet"


def load_dataset(parquet_url: str) -> pd.DataFrame:
    """Load parquet when available; fallback to CSV if parquet engine is missing."""
    try:
        return pd.read_parquet(parquet_url)
    except Exception:
        csv_url = parquet_url.replace(".parquet", ".csv")
        return pd.read_csv(csv_url)


# Load datasets
transaction_df = load_dataset(TRANSACTION_URL)
item_df = load_dataset(ITEM_URL)
premise_df = load_dataset(PREMISE_URL)

if "date" in transaction_df.columns:
    transaction_df["date"] = pd.to_datetime(transaction_df["date"], errors="coerce")

print("transaction_df:", transaction_df.shape)
print("item_df:", item_df.shape)
print("premise_df:", premise_df.shape)

transaction_df.head()

ImportError: Unable to find a usable engine; tried using: 'pyarrow', 'fastparquet'.
A suitable version of pyarrow or fastparquet is required for parquet support.
Trying to import the above resulted in these errors:
 - Missing optional dependency 'pyarrow'. pyarrow is required for parquet support. Use pip or conda to install pyarrow.
 - Missing optional dependency 'fastparquet'. fastparquet is required for parquet support. Use pip or conda to install fastparquet.

In [ ]:
# Metadata (from data.gov.my catalog pages)
metadata = {
    "pricecatcher": {
        "description": "Transactional price records.",
        "variables": ["date", "premise_code", "item_code", "price"],
        "url": TRANSACTION_URL,
        "join_key": ["premise_code", "item_code"],
    },
    "lookup_item": {
        "description": "Item lookup table.",
        "variables": ["item_code", "item_name", "unit", "item_group", "item_category"],
        "url": ITEM_URL,
        "join_key": ["item_code"],
    },
    "lookup_premise": {
        "description": "Premise lookup table.",
        "variables": ["premise_code", "premise", "address", "premise_type", "state", "district"],
        "url": PREMISE_URL,
        "join_key": ["premise_code"],
    },
}

metadata

In [ ]:
# Join to one raw table (left joins preserve all transactions)
joined_df = (
    transaction_df
    .merge(item_df, on="item_code", how="left", validate="many_to_one")
    .merge(premise_df, on="premise_code", how="left", validate="many_to_one")
)

print("joined_df:", joined_df.shape)
joined_df.head()

In [ ]:
# Save raw joined dataset (before cleaning/normalization)
os.makedirs("data", exist_ok=True)
raw_output_parquet = "data/pricecatcher_joined_raw.parquet"
raw_output_csv = "data/pricecatcher_joined_raw.csv"

try:
    joined_df.to_parquet(raw_output_parquet, index=False)
    print(f"Saved raw joined dataset to: {raw_output_parquet}")
except Exception:
    joined_df.to_csv(raw_output_csv, index=False)
    print(f"Parquet engine unavailable; saved CSV instead: {raw_output_csv}")

# Quick EDA snapshot
eda_summary = {
    "rows": len(joined_df),
    "columns": joined_df.shape[1],
    "duplicates": int(joined_df.duplicated().sum()),
}

print("EDA summary:", eda_summary)
print("\nDtypes:")
display(joined_df.dtypes.to_frame("dtype"))
print("\nMissing values (%):")
display((joined_df.isna().mean() * 100).sort_values(ascending=False).to_frame("missing_pct"))